In [3]:
# ============================================================
# PHASE 1 — DIRTY CSV INVESTIGATION
# ============================================================
#
# Goal:
# Profile the raw working-capital dataset before making any
# transformations or cleaning decisions.
#
# Business purpose:
# Establish the dataset's structure, potential grain, key fields,
# missingness, duplicates, categorical consistency, numeric
# integrity, and date quality.
#
# Important:
# This phase is investigative only. The raw data is not modified.
# ============================================================


# ============================================================
# 1. LOAD THE RAW DATASET
# ============================================================

import pandas as pd

# Load the raw CSV exactly as received.
# We keep the source data unchanged so every later transformation
# can be traced back to the original dataset.
df = pd.read_csv(r"C:\Users\hp\Downloads\Working_capital_dirty .csv")

print("Dataset loaded successfully.")


# ============================================================
# 2. DATASET STRUCTURE
# ============================================================

print("\n" + "=" * 60)
print("2. DATASET STRUCTURE")
print("=" * 60)

# Shape returns:
# (number of rows, number of columns)
print("Rows and columns:", df.shape)

# Display the column names so we know what business attributes
# are available for investigation.
print("\nColumns:")
for column in df.columns:
    print("-", column)

# Display the data types detected by Pandas.
# This helps identify fields that may need investigation, such as
# numeric or date fields being interpreted as text.
print("\nData types:")
print(df.dtypes)


# ============================================================
# 3. SAMPLE RECORDS
# ============================================================

print("\n" + "=" * 60)
print("3. SAMPLE RECORDS")
print("=" * 60)

# Display the first five records to understand what one raw
# record looks like before any transformations.
print(df.head())


# ============================================================
# 4. MISSING VALUE PROFILE
# ============================================================

print("\n" + "=" * 60)
print("4. MISSING VALUE PROFILE")
print("=" * 60)

# Count missing values in every column.
missing_count = df.isna().sum()

# Calculate the percentage of rows affected by missing values.
missing_percentage = (
    df.isna().mean() * 100
).round(2)

# Combine both measurements into one report.
missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_percentage
})

# Display only columns that contain missing values.
missing_report = missing_report[
    missing_report["missing_count"] > 0
].sort_values(
    "missing_count",
    ascending=False
)

print(missing_report)


# ============================================================
# 5. EXACT DUPLICATE ROWS
# ============================================================

print("\n" + "=" * 60)
print("5. EXACT DUPLICATE ROWS")
print("=" * 60)

# An exact duplicate means every column in the row is identical
# to another row.
exact_duplicate_count = df.duplicated().sum()

print("Exact duplicate rows:", exact_duplicate_count)


# ============================================================
# 6. POTENTIAL BUSINESS KEY — INVOICE_ID
# ============================================================

print("\n" + "=" * 60)
print("6. INVOICE_ID KEY INVESTIGATION")
print("=" * 60)

# Count the number of distinct invoice IDs.
unique_invoice_ids = df["invoice_id"].nunique()

# Count how many rows have a missing invoice ID.
missing_invoice_ids = df["invoice_id"].isna().sum()

print("Total rows:", len(df))
print("Unique invoice IDs:", unique_invoice_ids)
print("Missing invoice IDs:", missing_invoice_ids)

# Identify every row belonging to an invoice_id that appears
# more than once.
duplicate_invoice_rows = df[
    df["invoice_id"].duplicated(keep=False)
].sort_values("invoice_id")

print(
    "\nRows belonging to duplicated invoice IDs:",
    len(duplicate_invoice_rows)
)

# Count how many different invoice IDs are duplicated.
duplicated_invoice_ids = (
    duplicate_invoice_rows["invoice_id"].nunique()
)

print(
    "Number of duplicated invoice IDs:",
    duplicated_invoice_ids
)

# Show how many times each duplicated invoice ID occurs.
duplicate_frequency = (
    duplicate_invoice_rows["invoice_id"]
    .value_counts()
    .value_counts()
    .sort_index()
)

print("\nDuplicate frequency:")
print(
    duplicate_frequency.rename(
        "number_of_invoice_ids"
    ).rename_axis(
        "number_of_occurrences"
    )
)


# ============================================================
# 7. GRAIN INVESTIGATION
# ============================================================

print("\n" + "=" * 60)
print("7. GRAIN INVESTIGATION")
print("=" * 60)

# The grain describes what one row represents.
#
# We cannot simply assume that invoice_id is the grain.
# First we inspect how many records exist per invoice.
invoice_row_counts = (
    df.groupby("invoice_id", dropna=False)
      .size()
      .value_counts()
      .sort_index()
)

print(
    "Number of invoice IDs by number of rows:"
)
print(
    invoice_row_counts.rename(
        "number_of_invoice_ids"
    ).rename_axis(
        "rows_per_invoice"
    )
)

# Show a sample of duplicated invoice records.
# This lets us later determine whether repeated invoice IDs are
# exact duplicates or conflicting business records.
print("\nSample duplicated invoice records:")

print(
    duplicate_invoice_rows.head(20).to_string(index=False)
)


# ============================================================
# 8. CARDINALITY / UNIQUE VALUES
# ============================================================

print("\n" + "=" * 60)
print("8. COLUMN CARDINALITY")
print("=" * 60)

# Cardinality tells us how many distinct values each column contains.
# This is useful for identifying potential IDs, dimensions,
# categorical fields, and suspiciously high/low uniqueness.
cardinality_report = pd.DataFrame({
    "unique_values": df.nunique(dropna=True),
    "missing_values": df.isna().sum()
})

cardinality_report["unique_percentage"] = (
    cardinality_report["unique_values"] / len(df) * 100
).round(2)

print(cardinality_report.sort_values(
    "unique_values",
    ascending=False
))


# ============================================================
# 9. CATEGORICAL VALUE PROFILING
# ============================================================

print("\n" + "=" * 60)
print("9. CATEGORICAL VALUE PROFILING")
print("=" * 60)

categorical_columns = [
    "region",
    "currency",
    "payment_method",
    "status"
]

# Display the frequency of every value in important categorical
# fields exactly as they appear in the raw dataset.
#
# We deliberately do NOT strip whitespace or change case here.
# Those inconsistencies are part of what we are trying to discover.
for column in categorical_columns:

    print("\n" + "-" * 50)
    print(f"{column.upper()}")
    print("-" * 50)

    print(
        df[column]
        .value_counts(dropna=False)
    )


# ============================================================
# 10. NUMERIC FIELD INVESTIGATION
# ============================================================

print("\n" + "=" * 60)
print("10. NUMERIC FIELD INVESTIGATION")
print("=" * 60)

numeric_columns = [
    "invoice_amount",
    "payment_amount"
]

for column in numeric_columns:

    print("\n" + "-" * 50)
    print(f"{column.upper()}")
    print("-" * 50)

    # Show the raw data type.
    print("Pandas data type:", df[column].dtype)

    # Show the most common raw values.
    print("\nMost common raw values:")
    print(
        df[column]
        .value_counts(dropna=False)
        .head(15)
    )


# ============================================================
# 11. TEMPORARY NUMERIC CONVERSION FOR INVESTIGATION
# ============================================================

print("\n" + "=" * 60)
print("11. NUMERIC CONVERSION INVESTIGATION")
print("=" * 60)

# We create temporary numeric versions of the amount fields.
#
# IMPORTANT:
# The original columns are NOT modified.
#
# Removing commas allows values such as "12,500.00" to be
# interpreted as numbers.
#
# Values such as "N/A", "unknown", and "-" become NaN.
# That is useful for identifying malformed numeric values.

invoice_amount_numeric = pd.to_numeric(
    df["invoice_amount"]
    .astype("string")
    .str.replace(",", "", regex=False),
    errors="coerce"
)

payment_amount_numeric = pd.to_numeric(
    df["payment_amount"],
    errors="coerce"
)

# Count values that could not be interpreted as numbers.
# We exclude original missing values so that we distinguish
# "already missing" from "present but invalid".
invoice_invalid_numeric = (
    invoice_amount_numeric.isna()
    & df["invoice_amount"].notna()
).sum()

payment_invalid_numeric = (
    payment_amount_numeric.isna()
    & df["payment_amount"].notna()
).sum()

print(
    "Invoice amounts present but not convertible to numeric:",
    invoice_invalid_numeric
)

print(
    "Payment amounts present but not convertible to numeric:",
    payment_invalid_numeric
)

# Display valid numeric ranges.
print("\nInvoice amount range:")
print(
    invoice_amount_numeric.describe()
)

print("\nPayment amount range:")
print(
    payment_amount_numeric.describe()
)

# Investigate negative and zero values.
print("\nNegative invoice amounts:")
print(
    (invoice_amount_numeric < 0).sum()
)

print("Zero invoice amounts:")
print(
    (invoice_amount_numeric == 0).sum()
)

print("\nNegative payment amounts:")
print(
    (payment_amount_numeric < 0).sum()
)

print("Zero payment amounts:")
print(
    (payment_amount_numeric == 0).sum()
)


# ============================================================
# 12. DATE FIELD INVESTIGATION
# ============================================================

print("\n" + "=" * 60)
print("12. DATE FIELD INVESTIGATION")
print("=" * 60)

date_columns = [
    "invoice_date",
    "due_date",
    "payment_date"
]

for column in date_columns:

    print("\n" + "-" * 50)
    print(f"{column.upper()}")
    print("-" * 50)

    # Show the raw data type before conversion.
    print("Pandas data type:", df[column].dtype)

    # Temporarily convert the date values.
    # Invalid date strings become NaT rather than causing the
    # notebook to fail.
    temporary_dates = pd.to_datetime(
        df[column],
        errors="coerce"
    )

    # Count values that were present but could not be interpreted
    # as valid dates.
    invalid_dates = (
        temporary_dates.isna()
        & df[column].notna()
    ).sum()

    print(
        "Present but invalid dates:",
        invalid_dates
    )

    print(
        "Valid date range:",
        temporary_dates.min(),
        "to",
        temporary_dates.max()
    )

    print(
        "Unique valid dates:",
        temporary_dates.nunique()
    )


# ============================================================
# 13. INITIAL DATE BUSINESS-RELATIONSHIP INVESTIGATION
# ============================================================

print("\n" + "=" * 60)
print("13. INITIAL DATE RELATIONSHIPS")
print("=" * 60)

# Create temporary datetime representations.
invoice_dates = pd.to_datetime(
    df["invoice_date"],
    errors="coerce"
)

due_dates = pd.to_datetime(
    df["due_date"],
    errors="coerce"
)

payment_dates = pd.to_datetime(
    df["payment_date"],
    errors="coerce"
)

# Check whether due dates occur before invoice dates.
due_before_invoice = (
    due_dates < invoice_dates
).sum()

# Check whether payment dates occur before invoice dates.
payment_before_invoice = (
    payment_dates < invoice_dates
).sum()

print(
    "Due dates before invoice dates:",
    due_before_invoice
)

print(
    "Payment dates before invoice dates:",
    payment_before_invoice
)


# ============================================================
# 14. PAYMENT / STATUS RELATIONSHIP
# ============================================================

print("\n" + "=" * 60)
print("14. PAYMENT / STATUS RELATIONSHIP")
print("=" * 60)

# Compare payment-related fields with invoice status.
#
# This is still an investigation, not a cleaning rule.
# We want to discover whether the raw data follows an apparent
# relationship such as:
#
# Paid  -> payment date/payment amount expected
# Open  -> payment date/payment amount potentially absent

status_payment_profile = pd.crosstab(
    df["status"],
    [
        df["payment_date"].isna(),
        df["payment_amount"].isna()
    ]
)

print(status_payment_profile)


# ============================================================
# 15. POTENTIAL OVERPAYMENTS
# ============================================================

print("\n" + "=" * 60)
print("15. PAYMENT VS INVOICE AMOUNT")
print("=" * 60)

# Compare payment amounts with invoice amounts using the temporary
# numeric versions created earlier.
amount_comparison = pd.DataFrame({
    "invoice_amount": invoice_amount_numeric,
    "payment_amount": payment_amount_numeric
})

# Count records where payment exceeds invoice amount.
payment_greater_than_invoice = (
    amount_comparison["payment_amount"]
    > amount_comparison["invoice_amount"]
).sum()

print(
    "Rows where payment amount exceeds invoice amount:",
    payment_greater_than_invoice
)


# ============================================================
# 16. POTENTIAL KEY COLUMNS
# ============================================================

print("\n" + "=" * 60)
print("16. POTENTIAL KEY INVESTIGATION")
print("=" * 60)

potential_key_columns = [
    "invoice_id",
    "customer_id",
    "supplier_id",
    "product_id"
]

key_profile = pd.DataFrame({
    "rows": len(df),
    "unique_values": [
        df[column].nunique(dropna=True)
        for column in potential_key_columns
    ],
    "missing_values": [
        df[column].isna().sum()
        for column in potential_key_columns
    ]
}, index=potential_key_columns)

key_profile["unique_percentage"] = (
    key_profile["unique_values"]
    / len(df)
    * 100
).round(2)

print(key_profile)


# ============================================================
# 17. INITIAL INVESTIGATION SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("17. INITIAL INVESTIGATION SUMMARY")
print("=" * 60)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Exact duplicate rows:", exact_duplicate_count)
print("Unique invoice IDs:", unique_invoice_ids)
print("Duplicated invoice IDs:", duplicated_invoice_ids)
print("Missing invoice IDs:", missing_invoice_ids)
print(
    "Invalid invoice amounts:",
    invoice_invalid_numeric
)
print(
    "Invalid payment amounts:",
    payment_invalid_numeric
)
print(
    "Due dates before invoice dates:",
    due_before_invoice
)
print(
    "Payment dates before invoice dates:",
    payment_before_invoice
)
print(
    "Payments greater than invoice amounts:",
    payment_greater_than_invoice
)

print("\nPhase 1 investigation complete.")

Dataset loaded successfully.

2. DATASET STRUCTURE
Rows and columns: (12110, 14)

Columns:
- invoice_id
- customer_id
- customer_name
- region
- supplier_id
- product_id
- invoice_date
- due_date
- payment_date
- invoice_amount
- payment_amount
- currency
- payment_method
- status

Data types:
invoice_id         object
customer_id        object
customer_name      object
region             object
supplier_id        object
product_id         object
invoice_date       object
due_date           object
payment_date       object
invoice_amount     object
payment_amount    float64
currency           object
payment_method     object
status             object
dtype: object

3. SAMPLE RECORDS
   invoice_id customer_id         customer_name     region supplier_id  \
0  INV-001484       C0029     Tangier Logistics      Rabat        S006   
1  INV-009245       C0048       Agadir Commerce  Marrakesh        S003   
2  INV-008384       C0010        Fes Industries  Marrakesh        S002   
3  INV-00948

In [5]:
# ============================================================
# 18. DUPLICATE INVOICE CONSISTENCY CHECK
# ============================================================
#
# Goal:
# Determine whether duplicated invoice IDs represent exact
# duplicate records or conflicting versions of the same invoice.
#
# Why this matters:
# A repeated invoice_id is not automatically an error.
# We first need to know whether the repeated records contain
# identical or conflicting business information.
# ============================================================

# Select all rows belonging to duplicated invoice IDs.
duplicate_invoices = df[
    df["invoice_id"].duplicated(keep=False)
].copy()

# Compare all business attributes except invoice_id itself.
comparison_columns = [
    column
    for column in duplicate_invoices.columns
    if column != "invoice_id"
]

# For each duplicated invoice_id, count how many distinct
# versions of the remaining business attributes exist.
#
# 1 distinct version = all repeated records are identical.
# More than 1 = the same invoice_id has conflicting records.
duplicate_versions = (
    duplicate_invoices
    .groupby("invoice_id")[comparison_columns]
    .apply(
        lambda group: group.drop_duplicates().shape[0]
    )
    .rename("distinct_versions")
    .reset_index()
)

print("\nDistinct versions per duplicated invoice:")
print(
    duplicate_versions[
        "distinct_versions"
    ].value_counts().sort_index()
)

# Identify invoice IDs with conflicting information.
conflicting_invoice_ids = duplicate_versions.loc[
    duplicate_versions["distinct_versions"] > 1,
    "invoice_id"
]

print(
    "\nNumber of invoice IDs with conflicting records:",
    len(conflicting_invoice_ids)
)

# Display the conflicting records, if any.
if len(conflicting_invoice_ids) > 0:

    conflicting_duplicates = duplicate_invoices[
        duplicate_invoices["invoice_id"].isin(
            conflicting_invoice_ids
        )
    ].sort_values("invoice_id")

    print("\nConflicting duplicate records:")
    print(
        conflicting_duplicates.to_string(index=False)
    )

else:

    print(
        "\nNo conflicting duplicate invoice IDs found."
    )


Distinct versions per duplicated invoice:
distinct_versions
1    109
Name: count, dtype: int64

Number of invoice IDs with conflicting records: 0

No conflicting duplicate invoice IDs found.
